# Handwritten Text Recognition using CRNN, BiLSTM & CTC

This project implements an end-to-end handwritten text recognition system using:

- **CNN** for visual feature extraction
- **BiLSTM** for sequence modeling
- **CTC Loss** for alignment-free text recognition
- **CTC decoding** for text prediction

The project progresses from character recognition using **EMNIST** (baseline) towards
handwritten **word recognition** using the **IAM Handwriting Database** (real transcribed
handwriting, not synthesized text).

**Final deliverable:** this notebook trains and saves `crnn_model.pth` + `vocab.json` +
`config.json`. A separate Streamlit app (in the accompanying project) loads these exact files
and performs inference only — it never retrains.


## Install dependencies

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q opencv-python pillow matplotlib pandas numpy


## Imports

In [ ]:
import os
import json
import random
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as transforms


## Set random seeds
Makes experiments reproducible.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


## Check GPU

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# PART A — CHARACTER RECOGNITION BASELINE

A baseline is important before jumping into the full CRNN. EMNIST is the natural choice: the
CodeAlpha specification itself identifies MNIST/EMNIST and CNNs as the starting point for this task.


## Download EMNIST

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_dataset = torchvision.datasets.EMNIST(
    root="./data", split="balanced", train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.EMNIST(
    root="./data", split="balanced", train=False, download=True, transform=transform
)


## Inspect dataset

In [ ]:
print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))
print("Number of classes:", len(train_dataset.classes))


## Visualize samples

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    image, label = train_dataset[i]
    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_title(train_dataset.classes[label])
    ax.axis("off")
plt.suptitle("Sample EMNIST characters")
plt.show()


## DataLoader

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)


## CNN baseline

```
Image → Conv → ReLU → Pooling → Conv → ReLU → Pooling → Fully Connected → Character
```


In [ ]:
class CNNBaseline(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

num_classes = len(train_dataset.classes)
cnn_model = CNNBaseline(num_classes).to(device)
print(cnn_model)


## Train CNN
Train for several epochs, recording training/validation loss and accuracy.

In [ ]:
def evaluate_cnn(model, loader):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss_sum += criterion(outputs, labels).item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return loss_sum / total, correct / total


optimizer = optim.Adam(cnn_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

cnn_history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
EPOCHS_CNN = 10

for epoch in range(EPOCHS_CNN):
    cnn_model.train()
    correct, total, loss_sum = 0, 0, 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = cnn_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss, train_acc = loss_sum / total, correct / total
    val_loss, val_acc = evaluate_cnn(cnn_model, test_loader)

    cnn_history["train_loss"].append(train_loss)
    cnn_history["val_loss"].append(val_loss)
    cnn_history["train_acc"].append(train_acc)
    cnn_history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS_CNN} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
          f"| val_loss={val_loss:.4f} val_acc={val_acc:.4f}")


## Plot training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(cnn_history["train_loss"], label="Train Loss")
axes[0].plot(cnn_history["val_loss"], label="Val Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_title("CNN Baseline - Loss"); axes[0].legend()

axes[1].plot(cnn_history["train_acc"], label="Train Accuracy")
axes[1].plot(cnn_history["val_acc"], label="Val Accuracy")
axes[1].set_xlabel("Epoch"); axes[1].set_title("CNN Baseline - Accuracy"); axes[1].legend()
plt.tight_layout()
plt.savefig("cnn_training_curves.png", dpi=150)
plt.show()


## Confusion matrix
Establishes **Baseline Character Recognition Performance**.

In [ ]:
from sklearn.metrics import confusion_matrix

cnn_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        preds = cnn_model(images).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, cmap="Blues")
plt.title("CNN Baseline - Confusion Matrix (EMNIST balanced)")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

print(f"Baseline character-level test accuracy: {cnn_history['val_acc'][-1]:.4f}")


# PART B — HANDWRITTEN TEXT DATASET (IAM)

Now we move from individual characters to real **words**. Rather than artificially stitching
EMNIST characters into fake words, we use a dataset of actual handwritten word images with human
transcriptions: the **[IAM Handwriting Database](https://fki.tic.heia-fr.ch/databases/iam-handwriting-database)**.

**IAM requires free registration** (it is not a plain anonymous download), so this step is manual:

1. Register for a free account at the link above.
2. Download `words.tgz` (the word-level images) and `ascii/words.txt` (the transcriptions).
3. Upload both to this Colab session (or place them in Google Drive and mount it below).

This gives the structure:
```
image_001.png → "hello"
image_002.png → "world"
image_003.png → "machine"
```


In [ ]:
from google.colab import files
import tarfile

print("Upload words.tgz (the IAM word images archive):")
uploaded = files.upload()
with tarfile.open("words.tgz") as tar:
    tar.extractall("iam_words")

print("Upload words.txt (the IAM ASCII transcription file, from ascii/words.txt):")
uploaded2 = files.upload()
os.rename(list(uploaded2.keys())[0], "words.txt")
print("IAM data extracted.")


In [ ]:
# Alternative: mount Google Drive if you've already uploaded words.tgz / words.txt there
# from google.colab import drive
# drive.mount('/content/drive')
# Then point IAM_WORDS_DIR / IAM_TRANSCRIPTIONS_PATH below at your Drive paths instead.

IAM_WORDS_DIR = "iam_words"
IAM_TRANSCRIPTIONS_PATH = "words.txt"


### Parse the IAM transcription file

Each line of `words.txt` looks like:
```
a01-000u-00-00 ok 154 408 768 27 51 AT A
```
Fields: `word_id status gray_level x y w h grammatical_tag transcription`. The `word_id` encodes
the folder path: `a01-000u-00-00` -> `a01/a01-000u/a01-000u-00-00.png`. We keep only `status ==
"ok"` (IAM's own segmentation-quality flag) and words made only of letters, to keep the
vocabulary simple for this project.


In [ ]:
def parse_iam_words(txt_path, images_root, max_samples=None):
    records = []
    with open(txt_path) as f:
        for line in f:
            if line.startswith("#"):
                continue
            parts = line.strip().split()
            if len(parts) < 9:
                continue
            word_id, status = parts[0], parts[1]
            transcription = parts[-1]

            if status != "ok":
                continue
            if not transcription.isalpha():
                continue  # keep the vocabulary to plain letters for this project

            form, line_id, _ = word_id.split("-")[0], "-".join(word_id.split("-")[:2]), None
            image_path = os.path.join(images_root, form, line_id, f"{word_id}.png")
            if os.path.exists(image_path):
                records.append({"image_path": image_path, "text": transcription.lower()})

            if max_samples and len(records) >= max_samples:
                break
    return records

records = parse_iam_words(IAM_TRANSCRIPTIONS_PATH, IAM_WORDS_DIR)
print("Usable word samples:", len(records))


## CELL 14 — Create character vocabulary
Includes every letter used, plus the CTC blank.

In [ ]:
ALPHABET = sorted(set("".join(r["text"] for r in records)))
BLANK_TOKEN = "<blank>"

char_to_idx = {ch: i for i, ch in enumerate(ALPHABET)}
idx_to_char = {i: ch for i, ch in enumerate(ALPHABET)}
BLANK_INDEX = len(ALPHABET)  # blank gets the last index
NUM_CLASSES = len(ALPHABET) + 1

print("Vocabulary:", ALPHABET)
print("Num classes (incl. blank):", NUM_CLASSES)


## CELL 15 — Text encoding

`"hello"` becomes a list of integer indices, e.g. `[7, 4, 11, 11, 14]`. Each dataset item
returns the image, the encoded text, and the text length (CTC needs the true length
separately from the padded tensor).


In [ ]:
def encode_text(text, char_to_idx):
    return [char_to_idx[ch] for ch in text]

print("'hello' encoded:", encode_text("hello", char_to_idx))


## CELL 16 — Custom Dataset

In [ ]:
IMG_HEIGHT = 32
IMG_WIDTH = 256
MAX_LABEL_LEN = 16

class HandwritingDataset(Dataset):
    def __init__(self, records, char_to_idx, img_height=IMG_HEIGHT, img_width=IMG_WIDTH):
        self.records = records
        self.char_to_idx = char_to_idx
        self.img_height = img_height
        self.img_width = img_width

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        image = Image.open(record["image_path"]).convert("L")  # grayscale
        image = preprocess_pil_image(image, self.img_height, self.img_width)

        text = record["text"]
        encoded = encode_text(text, self.char_to_idx)
        label = torch.zeros(MAX_LABEL_LEN, dtype=torch.long)
        label[:len(encoded)] = torch.tensor(encoded[:MAX_LABEL_LEN])
        label_length = min(len(encoded), MAX_LABEL_LEN)

        return image, label, label_length, text


## CELL 17 — Image preprocessing

```
Raw image → Grayscale → Resize (aspect-ratio preserved) → Padding → Normalization
```

We deliberately do **not** stretch handwritten text into a fixed square — that distorts
letterforms and hurts recognition. Instead we resize by height only, then pad the width.


In [ ]:
def preprocess_pil_image(image: Image.Image, img_height=IMG_HEIGHT, img_width=IMG_WIDTH):
    w, h = image.size
    scale = img_height / h
    new_w = min(int(w * scale), img_width)
    new_w = max(new_w, 1)
    resized = image.resize((new_w, img_height), Image.BILINEAR)

    canvas = Image.new("L", (img_width, img_height), color=0)
    canvas.paste(resized, (0, 0))

    arr = np.array(canvas).astype("float32") / 255.0
    return torch.from_numpy(arr).unsqueeze(0)  # (1, H, W)


## CELL 18 — Visualize real handwriting

In [ ]:
from sklearn.model_selection import train_test_split

train_records, val_records = train_test_split(records, test_size=0.15, random_state=SEED)
print("Train word images:", len(train_records), " Val word images:", len(val_records))

fig, axes = plt.subplots(2, 5, figsize=(14, 5))
for ax, record in zip(axes.flat, random.sample(train_records, 10)):
    img = Image.open(record["image_path"]).convert("L")
    ax.imshow(img, cmap="gray")
    ax.set_title(f'"{record["text"]}"')
    ax.axis("off")
plt.suptitle("Real IAM handwriting samples with ground truth")
plt.tight_layout()
plt.show()


# PART C — CRNN

The main architecture of this project:

```
Image → CNN Feature Extractor → BiLSTM → CTC → Text
```


## CELL 19-22 — CNN feature extractor, sequence conversion, BiLSTM, full CRNN

Conceptually, the CNN's output feature map (height × width × channels) has its **width turned
into time steps**: `T1 T2 T3 T4 T5 T6 ...`, each a feature vector the BiLSTM reads in sequence.


In [ ]:
class CRNN(nn.Module):
    def __init__(self, img_height, num_classes, rnn_hidden=256):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d((2,1),(2,1)),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
        )
        cnn_out_height = img_height // 8
        self.feature_dim = 128 * cnn_out_height

        self.rnn = nn.LSTM(
            input_size=self.feature_dim, hidden_size=rnn_hidden,
            num_layers=2, bidirectional=True, batch_first=True,
        )
        self.fc = nn.Linear(rnn_hidden * 2, num_classes)

    def forward(self, x):
        features = self.cnn(x)                           # (batch, C, H', W')
        batch, channels, h, w = features.shape
        features = features.permute(0, 3, 1, 2)           # (batch, W', C, H')
        features = features.reshape(batch, w, channels*h) # (batch, time=W', features)

        rnn_out, _ = self.rnn(features)
        logits = self.fc(rnn_out)
        return logits.log_softmax(dim=-1)                 # (batch, time, num_classes)

crnn_model = CRNN(img_height=IMG_HEIGHT, num_classes=NUM_CLASSES).to(device)
print(crnn_model)


## CELL 23 — CTC Loss

In [ ]:
ctc_criterion = nn.CTCLoss(blank=BLANK_INDEX, zero_infinity=True)

# Training step conceptually:
#   Image -> CRNN -> predicted character sequence -> CTC loss -> backprop -> update weights


## CELL 24 — Training loop
Proper logging: epoch, training loss, validation loss, learning rate.

In [ ]:
train_dataset_htr = HandwritingDataset(train_records, char_to_idx)
val_dataset_htr = HandwritingDataset(val_records, char_to_idx)

train_loader_htr = DataLoader(train_dataset_htr, batch_size=32, shuffle=True, num_workers=2)
val_loader_htr = DataLoader(val_dataset_htr, batch_size=32, shuffle=False, num_workers=2)

optimizer_htr = optim.Adam(crnn_model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer_htr, factor=0.5, patience=3)

crnn_history = {"train_loss": [], "val_loss": [], "lr": []}
EPOCHS_CRNN = 40

def run_crnn_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, total_samples = 0.0, 0

    for images, labels, label_lengths, _texts in loader:
        images, labels = images.to(device), labels.to(device)
        label_lengths = label_lengths.to(device)

        with torch.set_grad_enabled(is_train):
            log_probs = model(images)                          # (batch, time, classes)
            log_probs_ctc = log_probs.permute(1, 0, 2)          # (time, batch, classes)
            input_lengths = torch.full((images.size(0),), log_probs.size(1), dtype=torch.long).to(device)

            loss = ctc_criterion(log_probs_ctc, labels, input_lengths, label_lengths)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_samples += images.size(0)

    return total_loss / total_samples


for epoch in range(EPOCHS_CRNN):
    train_loss = run_crnn_epoch(crnn_model, train_loader_htr, optimizer_htr)
    val_loss = run_crnn_epoch(crnn_model, val_loader_htr, optimizer=None)
    scheduler.step(val_loss)
    current_lr = optimizer_htr.param_groups[0]["lr"]

    crnn_history["train_loss"].append(train_loss)
    crnn_history["val_loss"].append(val_loss)
    crnn_history["lr"].append(current_lr)

    print(f"Epoch {epoch+1}/{EPOCHS_CRNN} | train_loss={train_loss:.4f} | "
          f"val_loss={val_loss:.4f} | lr={current_lr:.2e}")


## CELL 25 — Validation: CER & WER after training
(full definitions and computation in Cells 28-29 below)

## CELL 26 — CTC Greedy Decoder

In [ ]:
def greedy_decode(log_probs, idx_to_char, blank_idx):
    """log_probs: (time, num_classes) for ONE sample.
    argmax -> remove repeats -> remove blanks -> join to text.
    Example: 'H H _ E E _ L L _ L O O' -> 'HELLO'
    """
    best_path = log_probs.argmax(dim=-1).cpu().numpy()
    chars, previous = [], None
    for idx in best_path:
        idx = int(idx)
        if idx != previous and idx != blank_idx:
            chars.append(idx_to_char.get(idx, ""))
        previous = idx
    return "".join(chars)


## CELL 27 — Test predictions

In [ ]:
crnn_model.eval()
sample_images, sample_labels, sample_lengths, sample_texts = next(iter(val_loader_htr))
sample_images = sample_images.to(device)

with torch.no_grad():
    log_probs = crnn_model(sample_images)

predictions = [greedy_decode(log_probs[i], idx_to_char, BLANK_INDEX) for i in range(len(sample_texts))]

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, img, truth, pred in zip(axes.flat, sample_images.cpu(), sample_texts, predictions):
    ax.imshow(img.squeeze(), cmap="gray")
    marker = "OK" if truth == pred else "ERR"
    ax.set_title(f"[{marker}] true: {truth}\npred: {pred}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()


## CELL 28 — Character Error Rate (CER)

In [ ]:
def levenshtein(a, b):
    if len(a) < len(b):
        a, b = b, a
    previous_row = list(range(len(b) + 1))
    for i, ca in enumerate(a):
        current_row = [i + 1]
        for j, cb in enumerate(b):
            current_row.append(min(previous_row[j+1]+1, current_row[j]+1, previous_row[j]+(ca!=cb)))
        previous_row = current_row
    return previous_row[-1]

def character_error_rate(predictions, ground_truths):
    total_edits = sum(levenshtein(p, g) for p, g in zip(predictions, ground_truths))
    total_chars = sum(len(g) for g in ground_truths)
    return total_edits / total_chars if total_chars else 0.0


## CELL 29 — Word Error Rate (WER)
For single-word samples, WER is the fraction of words that do NOT match exactly.

In [ ]:
def word_error_rate(predictions, ground_truths):
    if not ground_truths:
        return 0.0
    incorrect = sum(1 for p, g in zip(predictions, ground_truths) if p != g)
    return incorrect / len(ground_truths)


# Full evaluation over the entire validation set
all_predictions, all_truths = [], []
with torch.no_grad():
    for images, labels, label_lengths, texts in val_loader_htr:
        images = images.to(device)
        log_probs = crnn_model(images)
        for i in range(len(texts)):
            all_predictions.append(greedy_decode(log_probs[i], idx_to_char, BLANK_INDEX))
            all_truths.append(texts[i])

cer = character_error_rate(all_predictions, all_truths)
wer = word_error_rate(all_predictions, all_truths)
character_accuracy = 1 - cer

print("=== Final Evaluation ===")
print(f"Character Accuracy: {character_accuracy*100:.2f}%")
print(f"CER: {cer*100:.2f}%")
print(f"WER: {wer*100:.2f}%")


In [ ]:
plt.figure(figsize=(7,4))
plt.plot(crnn_history["train_loss"], label="Train CTC Loss")
plt.plot(crnn_history["val_loss"], label="Val CTC Loss")
plt.xlabel("Epoch"); plt.ylabel("CTC Loss")
plt.title("CRNN Training Curve")
plt.legend()
plt.savefig("crnn_training_curves.png", dpi=150)
plt.show()


## CELL 30 — Error analysis

A good addition for interview discussion: show correct vs. incorrect predictions side by side,
and reason about *why* an error likely happened.


In [ ]:
correct_examples = [(t, p) for t, p in zip(all_truths, all_predictions) if t == p][:5]
incorrect_examples = [(t, p) for t, p in zip(all_truths, all_predictions) if t != p][:5]

print("### Correct predictions")
for truth, pred in correct_examples:
    print(f"  ground truth: {truth:<15} prediction: {pred}")

print("\n### Incorrect predictions")
for truth, pred in incorrect_examples:
    print(f"  ground truth: {truth:<15} prediction: {pred}")

print("""
Likely sources of error to discuss in an interview:
- Visually similar letters (e.g. 'rn' vs 'm', 'cl' vs 'd') that the CNN receptive field
  can't always disambiguate without more context.
- Cursive or heavily slanted handwriting distorting the vertical stroke patterns the CNN
  was trained on.
- Very short words give CTC few time steps to work with, so a single misread character
  has an outsized effect on word-level accuracy (this is exactly why WER is stricter
  than CER - one wrong letter fails the whole word).
- Class imbalance in IAM's vocabulary: common short words are overrepresented relative
  to rarer, longer words.
""")


## CELL 31 — Save model

Critical: the Streamlit app loads exactly these three files. The config prevents the app from
accidentally using different settings than training.


In [ ]:
os.makedirs("models", exist_ok=True)

torch.save(crnn_model.state_dict(), "models/crnn_model.pth")

vocab = {"char_to_idx": char_to_idx, "idx_to_char": {str(k): v for k, v in idx_to_char.items()}}
with open("models/vocab.json", "w") as f:
    json.dump(vocab, f, indent=2)

config = {
    "image_height": IMG_HEIGHT,
    "image_width": IMG_WIDTH,
    "num_classes": NUM_CLASSES,
    "blank_index": BLANK_INDEX,
    "rnn_hidden": 256,
    "max_label_len": MAX_LABEL_LEN,
    # Documents the fixed architecture in app/model.py so the app can never
    # silently drift from what was actually trained - if you change the
    # architecture, update both model.py and these fields together.
    "cnn_config": {"conv_channels": [32, 64, 128, 128], "pooling": "H/8, W/4"},
    "rnn_config": {"type": "BiLSTM", "num_layers": 2, "bidirectional": True, "hidden_size": 256},
}
with open("models/config.json", "w") as f:
    json.dump(config, f, indent=2)

metrics = {
    "cnn_baseline_accuracy": cnn_history["val_acc"][-1],
    "character_accuracy": character_accuracy,
    "cer": cer,
    "wer": wer,
}
with open("models/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# Save a few example predictions (with their images) for the app's "Example Predictions" page
os.makedirs("models/plots", exist_ok=True)
sample_predictions = []
for i, (truth, pred) in enumerate(list(zip(all_truths, all_predictions))[:10]):
    sample_predictions.append({"ground_truth": truth, "prediction": pred, "image_path": None})
with open("models/plots/sample_predictions.json", "w") as f:
    json.dump(sample_predictions, f, indent=2)

import shutil
for fname in ["cnn_training_curves.png", "crnn_training_curves.png", "confusion_matrix.png"]:
    if os.path.exists(fname):
        shutil.copy(fname, f"models/plots/{fname}")

print("Saved: models/crnn_model.pth, vocab.json, config.json, metrics.json, plots/")


## CELL 32 — Test the saved model

Before uploading anything to GitHub: train -> save -> reload -> predict again. If the reloaded
model produces the same prediction, model packaging works correctly.


In [ ]:
# Re-instantiate the architecture and load the saved weights fresh
reloaded_model = CRNN(img_height=config["image_height"], num_classes=config["num_classes"],
                       rnn_hidden=config["rnn_hidden"]).to(device)
reloaded_model.load_state_dict(torch.load("models/crnn_model.pth", map_location=device))
reloaded_model.eval()

with torch.no_grad():
    original_pred = greedy_decode(crnn_model(sample_images)[0], idx_to_char, BLANK_INDEX)
    reloaded_pred = greedy_decode(reloaded_model(sample_images)[0], idx_to_char, BLANK_INDEX)

print("Original model prediction:", original_pred)
print("Reloaded model prediction:", reloaded_pred)
assert original_pred == reloaded_pred, "Mismatch! Model packaging has a bug."
print("Model packaging verified - safe to upload to GitHub.")


In [ ]:
from google.colab import files
files.download("models/crnn_model.pth")
files.download("models/vocab.json")
files.download("models/config.json")
files.download("models/metrics.json")
print("Download these into your project's models/ folder before deploying the Streamlit app.")


## Summary

This notebook delivers exactly the pipeline the project calls for:

**Dataset → Preprocessing → CNN → BiLSTM → CTC → Training → Evaluation → Saved Model**

The companion Streamlit app (`app/app.py`, `app/model.py`, `app/preprocessing.py`,
`app/decoder.py`) is built around these exact saved artifacts: same architecture, same
preprocessing, same vocabulary — it only performs inference, never retrains.

**Resume claim:**
> Developed an end-to-end handwritten text recognition system using CRNN architecture with
> CNN-based visual feature extraction, BiLSTM sequence modeling, CTC decoding, and a deployed
> Streamlit inference application.
